# The hand-computed annual grant

An annual award sizes one grant per employee through the award calculator's seven steps
(`hcsim.financial.calculator`). This notebook follows one employee through every step by hand,
then checks each against what the engine's annual sizer (`AnnualGapFill`) computes. Every
number here is invented for the example.

The employee is at level 2, rated 3, and already vests shares from an earlier grant: 150 in the
coming year and 90 in the year after. The annual award is granted at step 12, and covers four
forward years, vesting a quarter of each year's shares every three months.

In [ ]:
import numpy as np
import pandas as pd

from hcsim.financial.calculator import RatingTable
from hcsim.financial.prices import AnnualGrowthPrice
from hcsim.financial.ranges import RangeTable
from hcsim.financial.sizing import AnnualGapFill, SizingContext
from hcsim.financial.tracker import HolderLedger
from hcsim.financial.vesting import quarterly
from hcsim.model.core.layouts import EMPLOYEE, EmployeeColumn
from hcsim.model.core.schema import (
    AWARD_TYPE,
    COST_CENTER,
    JOB_FAMILY,
    JOB_LEVEL,
    LOCATION,
    SimulationSchema,
)

schema = SimulationSchema.build(
    {
        COST_CENTER: ["CC-01", "CC-02"],
        JOB_FAMILY: ["F"],
        JOB_LEVEL: [1, 2],
        LOCATION: ["L"],
        AWARD_TYPE: ["annual"],
    }
)
ranges = RangeTable.from_frame(
    pd.DataFrame(
        [
            {
                "job_family": "F",
                "job_level": level,
                "location": "L",
                "min_base": base,
                "max_base": base + 40_000,
                "min_target": target,
                "max_target": target + 40_000,
                "base_adj": 1.0,
            }
            for level, base, target in ((1, 80_000, 120_000), (2, 100_000, 160_000))
        ]
    ),
    schema,
)
ratings = RatingTable({1: 0.25, 2: 0.5, 3: 0.75, 4: 1.0})
sizer = AnnualGapFill(ranges, quarterly("annual_quarterly", 4), ratings)

employee = EMPLOYEE.empty(1)
employee[0, EmployeeColumn.EMPLOYEE_ID] = 100001
employee[0, EmployeeColumn.LEVEL] = 1  # level 2, the vocabulary's second
employee[0, EmployeeColumn.PERF_RATING] = 3

# The earlier grant, awarded in month 0: its vests in months 15 to 24 fall in the annual
# award's first forward year, and those in months 27 to 36 in its second.
ledger = HolderLedger()
earlier = np.zeros((1, 72), dtype=np.int64)
earlier[0, [14, 17, 20, 23]] = [30, 40, 40, 40]
earlier[0, [26, 29, 32, 35]] = [20, 20, 25, 25]
ledger.record(np.array([0]), 0, earlier, 0, np.array([1]), np.array([0]))

price = AnnualGrowthPrice(100.0, growth_rate=1.25)
context = SizingContext(12, 1, ledger)
sized = sizer.sized(employee, price, context)

## Steps 1 to 4: range, penetration, target, and base

1. **Range.** The employee's cell is family F, level 2, location L: base 100,000 to 140,000,
   and target total compensation 160,000 to 200,000.
2. **Penetration.** The rating table puts rating 3 three quarters of the way into the range.
3. **Target.** `160,000 + 0.75 x (200,000 - 160,000) = 190,000`.
4. **Base**, imputed at the same penetration: `100,000 + 0.75 x (140,000 - 100,000) = 130,000`.

So the employee's equity target is 60,000 a year: target less base.

In [ ]:
penetration = 0.75
target = 160_000 + penetration * (200_000 - 160_000)
base = 1.0 * (100_000 + penetration * (140_000 - 100_000))
assert (target, base) == (190_000, 130_000)
assert sized.penetration.tolist() == [penetration]
assert sized.target.tolist() == [target]
assert sized.base.tolist() == [base]

## Step 5: each forward year's gap

The price grows by a quarter a year, so forward year *y* is projected at `100 x 1.25^y`: 125,
156.25, 195.3125, and 244.140625. Each year's gap is the equity target less the value of what
the employee already vests that year, never below zero:

| Year | Price | Already vesting | Gap |
|---|---|---|---|
| 1 | 125 | 150 | 60,000 - 150 x 125 = 41,250 |
| 2 | 156.25 | 90 | 60,000 - 90 x 156.25 = 45,937.50 |
| 3 | 195.3125 | 0 | 60,000 |
| 4 | 244.140625 | 0 | 60,000 |

In [ ]:
prices = [100 * 1.25**y for y in (1, 2, 3, 4)]
existing = [150, 90, 0, 0]
gaps = [max(0.0, target - base - held * p) for held, p in zip(existing, prices, strict=True)]
assert gaps == [41_250, 45_937.5, 60_000, 60_000]
assert sized.gap.tolist() == [gaps]

## Steps 6 and 7: shares, then whole shares

Each year's shares are its gap over its price: 330, 294, 307.2, and 245.76. Rounded to whole
shares, that's 330, 294, 307, and 246.

The quarterly template vests a quarter of each year's shares at months 3, 6, 9, and 12 of the
year. Each quarter takes the whole shares below its quarter, and the year's last vest takes what
remains. Year 1 vests 82 three times, then 330 - 3 x 82 = 84.

In [ ]:
shares_float = [gap / p for gap, p in zip(gaps, prices, strict=True)]
assert np.allclose(shares_float, [330, 294, 307.2, 245.76])
shares = [round(s) for s in shares_float]
assert shares == [330, 294, 307, 246]
assert sized.shares.tolist() == [shares]

vests = {}
for year, total in enumerate(shares):
    quarter = total // 4
    for k, month in enumerate((3, 6, 9, 12)):
        vests[12 * year + month] = quarter if k < 3 else total - 3 * quarter
schedule = sizer.size(employee, price, context).vest_schedule[0]
assert {int(m) + 1: int(schedule[m]) for m in np.flatnonzero(schedule)} == vests
vests